In [1]:
import sys
from pathlib import Path
from uncertainties import ufloat
import country_converter as coco
import pandas as pd
import math

# ------------------------ Run from Repo Root ------------------------
BASE_DIR = Path.cwd().parent
sys.path.append(str(BASE_DIR))

# ------------------------ File Paths ------------------------
CR_Box_Countries = BASE_DIR / "data" / "CR_Box_Countries_MS.csv"
country_list_csv = BASE_DIR / "data" / "countries_MS.csv"

from country_pkg import Country

In [2]:
# ------------------------ Functions ------------------------
def generate_countries(country_csv_path, cr_box_csv_path=None):
    """
    Generate a dictionary of Country objects from a CSV of country names.
    Optionally load CR_Box properties from a separate CSV.
    
    Returns:
        dict: {country_name: Country instance}
    """
    # Load the CSV with the list of countries
    df = pd.read_csv(country_csv_path)
    
    if 'Country' not in df.columns:
        raise ValueError("CSV must have a column named 'Country'")
    
    countries = {}
    for name in df['Country']:
        c = Country(name)
        if cr_box_csv_path:
            c.load_properties_from_csv(cr_box_csv_path, country_col="Country")
        countries[c.name] = c
    return countries

In [3]:
## ------------------------ CR Box Reference ------------------------ 

Ind_Market_Rev_Per_MERV = {
    '17-20' : 2208.7e6,
    '5-8'   : 563.4e6,
    '9-12'  : 1271.8e6,
    '1-4'   : 171.7e6,
    '13-16' : 1878.1e6
}

Tot_Ind_Air_Filter = Ind_Market_Rev_Per_MERV['17-20']+Ind_Market_Rev_Per_MERV['5-8']+Ind_Market_Rev_Per_MERV['1-4']+Ind_Market_Rev_Per_MERV['9-12']+Ind_Market_Rev_Per_MERV['13-16']

Tot_Air_Filter = 20.8303e9

All_Market_Rev_Per_MERV = {
    '17-20' : Ind_Market_Rev_Per_MERV['17-20']*Tot_Air_Filter/Tot_Ind_Air_Filter,
    '5-8'   : Ind_Market_Rev_Per_MERV['5-8']*Tot_Air_Filter/Tot_Ind_Air_Filter,
    '9-12'  : Ind_Market_Rev_Per_MERV['9-12']*Tot_Air_Filter/Tot_Ind_Air_Filter,
    '1-4'   : Ind_Market_Rev_Per_MERV['1-4']*Tot_Air_Filter/Tot_Ind_Air_Filter,
    '13-16' : Ind_Market_Rev_Per_MERV['13-16']*Tot_Air_Filter/Tot_Ind_Air_Filter,
}

Price_Per_Filter = {
    '1-4'   : ufloat(1031.59,   107.26/2),
    '5-8'   : ufloat(1133.85,   447.48/2),
    '9-12'  : ufloat(1302.51,   554.53/2),
    '13-16' : ufloat(1951.25,   593.63/2),
    '17-20' : ufloat(22925.29,  3740.81/2)
}

Volume_to_Sale = 0.508*0.508*0.0254

Sales = {
    '1-4'   : All_Market_Rev_Per_MERV['1-4']/(Price_Per_Filter['1-4']*Volume_to_Sale),
    '5-8'   : All_Market_Rev_Per_MERV['5-8']/(Price_Per_Filter['5-8']*Volume_to_Sale),
    '9-12'  : All_Market_Rev_Per_MERV['9-12']/(Price_Per_Filter['9-12']*Volume_to_Sale),
    '13-16' : All_Market_Rev_Per_MERV['13-16']/(Price_Per_Filter['13-16']*Volume_to_Sale),
    '17-20' : All_Market_Rev_Per_MERV['17-20']/(Price_Per_Filter['17-20']*Volume_to_Sale)
}

Panel_Filter = ufloat(0.35,     0.35*0.1/2)
Scale_Up_Factor = 1/0.7

Usable_Filters = (Sales['13-16']+Sales['17-20']) * Panel_Filter * Scale_Up_Factor

In [5]:
## ------------------------ Countries ------------------------ 

if "countries_dict" not in globals():
    countries_dict = generate_countries(country_list_csv, CR_Box_Countries)

sum_scale = 0
for country in countries_dict.values():
    msa = country.properties.get("Manufacturing Score Adjusted",0)
    mva = country.properties.get("MVA", 0)
    if msa == 1:
        sum_scale += mva
scale = Usable_Filters/sum_scale
print(scale)
for country in countries_dict.values():
    msa = country.properties.get("Manufacturing Score Adjusted",0)
    mva = country.properties.get("MVA", 0)
    x = scale * msa * mva
    # if x.nominal_value < 50000:
    #     x = 0
    # else:
    #     x = ufloat(math.floor(x.nominal_value), x.std_dev)
    country.properties["CR Box"] = x

for country in countries_dict.values():
    msa = country.properties.get("Manufacturing Score Adjusted",0)
    mva = country.properties.get("MVA", 0)
    print(country.summary)

(2.38+/-0.35)e-05
<bound method Country.summary of <Country Bahrain, properties: ['Country', 'Manufacturing Score', 'Manufacturing Score Adjusted', 'MVA', 'Relative MVA', 'CR Box']>>
<bound method Country.summary of <Country China, properties: ['Country', 'Manufacturing Score', 'Manufacturing Score Adjusted', 'MVA', 'Relative MVA', 'CR Box']>>
<bound method Country.summary of <Country Georgia, properties: ['Country', 'Manufacturing Score', 'Manufacturing Score Adjusted', 'MVA', 'Relative MVA', 'CR Box']>>
<bound method Country.summary of <Country India, properties: ['Country', 'Manufacturing Score', 'Manufacturing Score Adjusted', 'MVA', 'Relative MVA', 'CR Box']>>
<bound method Country.summary of <Country Indonesia, properties: ['Country', 'Manufacturing Score', 'Manufacturing Score Adjusted', 'MVA', 'Relative MVA', 'CR Box']>>
<bound method Country.summary of <Country Iran, properties: ['Country', 'Manufacturing Score', 'Manufacturing Score Adjusted', 'MVA', 'Relative MVA', 'CR Box']